# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates loading, exploration, and processing of the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

[https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset title and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will inspect the record sets defined in the dataset schema using their `@id` fields.

In [ ]:
# List all record sets and their @id fields
record_sets = []
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
else:
    # fallback: try single recordSet
    record_sets = getattr(metadata, 'recordSet', [])

print("Record Sets (@id):")
for rs in record_sets:
    print(f"- {rs['@id']}")

# For demonstration, if record_sets is empty, we can fetch from dataset directly
if not record_sets:
    from pprint import pprint
    record_sets = list(dataset.available_record_sets())
    print("Fetched available record sets:")
    pprint(record_sets)
    # Use the first for further exploration
if len(record_sets) > 0:
    record_set_id = record_sets[0]['@id'] if isinstance(record_sets[0], dict) else record_sets[0]
else:
    record_set_id = None

# Print a preview of records from the first record set
if record_set_id:
    records_preview = []
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        records_preview.append(record)
        if i >= 2:
            break
    print(f"Preview records from record set {record_set_id}:")
    for rec in records_preview:
        print(rec)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We will use the `@id` of the record set identified above to load its records.

In [ ]:
# For demonstration, extract the record set IDs from above
if isinstance(record_sets[0], dict):
    record_set_ids = [rs['@id'] for rs in record_sets]
else:
    record_set_ids = [rs for rs in record_sets]

# If record_set_ids still empty, fetch from dataset
if not record_set_ids:
    record_set_ids = list(dataset.available_record_sets())

dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df

if len(dataframes) > 0:
    main_rs_id = list(dataframes.keys())[0]
    print(f"Columns for record set {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head(3))
else:
    print("No records loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, grouping.

Let's select numeric and categorical fields by their `@id`, filter and normalize, and group.

In [ ]:
# Choose numeric and grouping fields (by @id) from available columns
df = dataframes.get(main_rs_id)
if df is not None:
    # Try to auto-select fields based on data
    numeric_fields = [col for col in df.columns if df[col].dtype in ['float64', 'int64']]
    group_fields = [col for col in df.columns if df[col].dtype == 'object']
    print(f"Numeric fields: {numeric_fields}")
    print(f"Group fields: {group_fields}")

    # Select a numeric field, e.g. 'GAD7_score' or 'PHQ9_score' if available
    if 'GAD7_score' in numeric_fields:
        numeric_field_id = 'GAD7_score'
    elif len(numeric_fields) > 0:
        numeric_field_id = numeric_fields[0]
    else:
        numeric_field_id = None

    # Threshold for filtering GAD7_score
    threshold = 10
    if numeric_field_id is not None:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head(3))

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head(3))
    else:
        print("No numeric field available for EDA.")

    group_field_id = None
    # Try grouping by available categorical fields
    for field in ['level_of_education', 'gender', 'village', 'marital_status']:
        if field in df.columns:
            group_field_id = field
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        display(grouped_df.head())
    else:
        print("No appropriate categorical field found for grouping.")
else:
    print("No dataframe available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Let's plot the distribution of a numeric field and compare across groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id is not None:
    # Plot histogram for numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    # Boxplot by category if grouping field exists
    if group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Mental Health Survey Dataset provides detailed survey responses from Kilifi County, Kenya, including demographic and standardized assessment scores (e.g. GAD-7).
- Exploratory analysis demonstrates how scores distribute across the population and can be grouped by categorical variables (e.g., education level).
- Further analysis can inform local interventions and help improve mental health survey design and data infrastructure.